# Eff-3DPSeg (spconv) on Google Colab

Before running: **Runtime -> Change runtime type -> GPU (T4)**. spconv needs CUDA.

Pheno4D comes from your Kaggle dataset. Create a Kaggle API token (Kaggle -> Settings -> API), then in Colab open **Secrets** (key icon in the left bar), add a secret named `KAGGLE_API_TOKEN` with the token as its value, and turn **Notebook access** on. Never paste the token into a cell or share it.

In [ ]:
# ---- edit these ----
REPO_URL = 'https://github.com/gitmate21/3d-phenomics.git'
KAGGLE_DATASET = 'hsenagb/pheno4d-clone'      # owner/name of your Kaggle dataset
# --------------------
import os, torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available())
if not torch.cuda.is_available() or torch.version.cuda is None:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> GPU, then Runtime -> Restart session.')

# Local session storage: fast, but wiped when the Colab runtime is recycled.
# Download the checkpoint (last cell) before the session ends.
CKPT_DIR = '/content/checkpoints'
LOG_DIR  = '/content/logs'
PRED_DIR = '/content/preds'
for d in (CKPT_DIR, LOG_DIR, PRED_DIR):
    os.makedirs(d, exist_ok=True)

In [ ]:
# Clone, or update an existing clone, then enter the project folder.
REPO = '/content/3d-phenomics'
if os.path.isdir(REPO):
    !git -C {REPO} pull origin main
else:
    !git clone {REPO_URL} {REPO}
%cd {REPO}/eff3dpseg

In [ ]:
# Sparse-conv backend. spconv has no working build for Colab's CUDA 13 (its C++ build fails),
# so use the pure-PyTorch implementation in scripts/sparse_torch.py: no install, any CUDA, a bit slower.
# Later cells run `!python scripts/...` which inherits this environment variable.
os.environ['EFF_SPARSE_BACKEND'] = 'torch'     # 'auto' tries spconv first, then falls back
!python scripts/test_sparse_torch.py

In [ ]:
# Download Pheno4D from Kaggle (needs the KAGGLE_API_TOKEN Colab secret described above).
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
!pip install -q -U kaggle
RAW_ROOT = '/content/pheno4d_raw'
if not os.path.isdir(RAW_ROOT):
    !kaggle datasets download -d {KAGGLE_DATASET} -p {RAW_ROOT} --unzip
!ls {RAW_ROOT}

In [ ]:
# Point RAW_DIR at the folder that contains one sub-folder per plant (look at the listing above).
RAW_DIR = RAW_ROOT + '/Pheno4D'
!ls {RAW_DIR}
!ls {RAW_DIR}/$(ls {RAW_DIR} | head -1) | head

In [ ]:
!python scripts/smoke_test.py

In [ ]:
# Convert raw txt -> npz + plant-level splits. Hold out whole plants for val/test:
# edit the plant names to ones that exist in the listing above.
DATA = '/content/pheno4d_npz'
!python scripts/prepare_pheno4d.py --raw {RAW_DIR} --out {DATA} --val-plants Tomato06 --test-plants Tomato07

In [ ]:
# --voxel-size is in the raw point-cloud units; the original used 0.2. Check your units!
# --resume auto continues from /content/checkpoints, which only survives while this runtime is alive.
!python scripts/train.py --data {DATA} --run-name demo --max-iter 2000 --validate-step 50 \
    --ckpt-dir {CKPT_DIR} --log-dir {LOG_DIR} --resume auto --time-limit-hours 10

In [ ]:
!python scripts/infer.py --data {DATA} --ckpt {CKPT_DIR}/demo_best.pth --phase test \
    --out-dir {PRED_DIR} --save-ply

# Local storage is wiped when the runtime ends: download results before closing.
!zip -qr /content/results.zip {CKPT_DIR} {LOG_DIR} {PRED_DIR}
from google.colab import files
files.download('/content/results.zip')